<a href="https://colab.research.google.com/github/RatchanonPa/Data-Warehouse-and-Big-Data-Analytics/blob/main/ASR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### ✅ Step-by-step: Transcribe a Porjai Sample with `Pathumma-whisper-th-large-v3`

#### 1. **Load the dataset (first 1,000 rows, streaming)**

```python
from datasets import load_dataset
from itertools import islice

dataset_streaming = load_dataset("CMKL/Porjai-Thai-voice-dataset-central", split="train", streaming=True)
first_1000 = list(islice(dataset_streaming, 1000))
```

#### 2. **Extract an audio sample**

```python
# We'll use the first audio sample
sample = first_1000[0]
audio_path = sample['audio']['path']  # Local path to audio file
print("Using audio file:", audio_path)
```

#### 3. **Resample the audio to 16kHz (if needed)**

```python
import torchaudio

waveform, sr = torchaudio.load(audio_path)
if sr != 16000:
    waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(waveform)
    torchaudio.save("temp_16k.wav", waveform, 16000)
    audio_path_16k = "temp_16k.wav"
else:
    audio_path_16k = audio_path
```

#### 4. **Load the Whisper pipeline**

```python
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

pipe = pipeline(
    task="automatic-speech-recognition",
    model="nectec/Pathumma-whisper-th-large-v3",
    torch_dtype=torch_dtype,
    device=device
)

pipe.model.config.forced_decoder_ids = pipe.tokenizer.get_decoder_prompt_ids(language="th", task="transcribe")
```

#### 5. **Transcribe the audio**

```python
transcription = pipe(audio_path_16k)["text"]
print("✅ Transcription:", transcription)
print("📌 Ground truth sentence:", sample['sentence'])
```

---

### 📝 Example Output

```text
✅ Transcription: สวัสดีค่ะ ดิฉันชื่อพอใจ
📌 Ground truth sentence: สวัสดีค่ะ ดิฉันชื่อพอใจ
```



# Start

In [1]:
pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pla

In [2]:
from datasets import load_dataset
from itertools import islice

# โหลดแบบ streaming แทนการโหลดทั้งหมดลง local
dataset_streaming = load_dataset("CMKL/Porjai-Thai-voice-dataset-central", split="train", streaming=True)

# ดึงมาแค่ 1000 แถวแรก
first_1000 = list(islice(dataset_streaming, 1000))

# แปลงเป็น Dataset แบบทั่วไป (optional)
from datasets import Dataset
dataset_1000 = Dataset.from_list(first_1000)

# แสดงข้อมูล
print(dataset_1000)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.17k [00:00<?, ?B/s]

Dataset({
    features: ['audio', 'sentence', 'utterance'],
    num_rows: 1000
})


In [3]:
from datasets import Audio

# แปลงคอลัมน์ 'audio' ให้สามารถเข้าถึง metadata ได้ (decode)
dataset_1000 = dataset_1000.cast_column("audio", Audio())

# ตรวจสอบค่า sampling rate ของแถวแรก
print("Sampling rate:", dataset_1000[0]["audio"]["sampling_rate"], "Hz")

Sampling rate: 16000 Hz


In [4]:
from datasets import Audio

# Decode the 'audio' column
dataset_1000 = dataset_1000.cast_column("audio", Audio())
dataset_1000

Dataset({
    features: ['audio', 'sentence', 'utterance'],
    num_rows: 1000
})

In [5]:
# Already casted: dataset_1000 = dataset_1000.cast_column("audio", Audio())

# Get the first sample
first_sample = dataset_1000[0]
audio_array = first_sample["audio"]["array"]
sampling_rate = first_sample["audio"]["sampling_rate"]

In [6]:
first_sample = dataset_1000[0]

print("Sentence:", first_sample["sentence"])
print("Utterance:", first_sample["utterance"])
print("Sampling Rate:", first_sample["audio"]["sampling_rate"], "Hz")
print("Audio Array Shape:", first_sample["audio"]["array"].shape)

Sentence: ทีม จาก อิสราเอล ไม่ ควร ได้ เป็น เจ้าบ้าน ใน เกม ยูฟ่า คัพ
Utterance: thai-central_000000
Sampling Rate: 16000 Hz
Audio Array Shape: (102400,)


In [7]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

pipe = pipeline(
    task="automatic-speech-recognition",
    model="nectec/Pathumma-whisper-th-large-v3",
    torch_dtype=torch_dtype,
    device=device
)

pipe.model.config.forced_decoder_ids = pipe.tokenizer.get_decoder_prompt_ids(language="th", task="transcribe")

config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/3.93k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Device set to use cuda:0


In [8]:
from IPython.display import Audio

# Play the audio
Audio(first_sample["audio"]["array"], rate=first_sample["audio"]["sampling_rate"])


In [9]:
# Whisper expects sampling rate = 16,000 Hz
if sampling_rate != 16000:
    import torchaudio
    import torch

    # Resample if needed
    waveform = torch.tensor(audio_array).unsqueeze(0)
    resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
    audio_array = resampler(waveform).squeeze().numpy()

# Transcribe
result = pipe({"array": audio_array, "sampling_rate": 16000})
print("🔊 Transcription:", result["text"])

/usr/local/lib/python3.11/dist-packages/transformers/models/whisper/generation_whisper.py:573: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(


🔊 Transcription: ทีม จาก อิสราเอล ไม่ ควร ได้ เป็น เจ้าบ้าน ใน เกม ยูฟ่า คัพ


In [10]:
# prompt: show time usage while inference

%%time
# Transcribe
result = pipe({"array": audio_array, "sampling_rate": 16000})
print("🔊 Transcription:", result["text"])


🔊 Transcription: ทีม จาก อิสราเอล ไม่ ควร ได้ เป็น เจ้าบ้าน ใน เกม ยูฟ่า คัพ
CPU times: user 8.77 s, sys: 120 ms, total: 8.89 s
Wall time: 8.81 s
